In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt")        # tokenizer
nltk.download("wordnet")      # lemmatizer data
nltk.download("stopwords")
from sklearn.metrics import accuracy_score,precision_score,recall_score

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#load the dataset
df=pd.read_csv("emails.csv")
df.rename(columns={"spam": "target"}, inplace=True)
df

,text,target
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
...,...,...
5723,Subject: re : research and development charges...,0
5724,"Subject: re : receipts from visit jim , than...",0
5725,Subject: re : enron case study update wow ! a...,0
5726,"Subject: re : interest david , please , call...",0


In [3]:
#infprmation about data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   target  5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


In [4]:
#checking shape of the data
df.shape

(5728, 2)

In [5]:
#checking null values
df.isnull().sum()

text      0
target    0
dtype: int64

In [6]:
import re
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [7]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


In [8]:
#text preprocessing
def preprocessing(text):
    text = str(text).lower()

    # remove the literal word 'subject'
    text = re.sub(r"\bsubject\b", "", text)

    # remove URLs, emails, numbers
    text = re.sub(r"http\S+|www\S+|\S+@\S+|\d+", "", text)

    tokens = word_tokenize(text)

    text_stop = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    lemma = [lemmatizer.lemmatize(word) for word in text_stop]
    return " ".join(lemma)

In [9]:
df["preprocess_text"]=df["text"].apply(preprocessing)
df["preprocess_text"]

0       naturally irresistible corporate identity lt r...
1       stock trading gunslinger fanny merrill muzo co...
2       unbelievable new home made easy im wanting sho...
3       color printing special request additional info...
4       money get software cd software compatibility g...
                              ...                        
5723    research development charge gpg forwarded shir...
5724    receipt visit jim thanks invitation visit lsu ...
5725    enron case study update wow day super thank mu...
5726    interest david please call shirley crenshaw as...
5727    news aurora update aurora version fastest mode...
Name: preprocess_text, Length: 5728, dtype: object

In [11]:
#defining X-independent variable  and y-dependent variable
X=df["text"]
y=df["target"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
#change text to numerical using tfidf vectorizer
text_vec=TfidfVectorizer()
X_train_vec=text_vec.fit_transform(X_train)
X_test_vec=text_vec.transform(X_test)

In [13]:
#balancing the model
from imblearn.over_sampling import RandomOverSampler
ros=RandomOverSampler()
X_train_res,y_train_res=ros.fit_resample(X_train_vec,y_train)

#t
raining the model
🥇 BEST OVERALL (classic choice)
Multinomial Naive Bayes

✅ Top choice for beginners & real projects

Why it works so well for spam:

Designed for text + word counts

Very fast

Works amazingly with TF-IDF / Bag-of-Words

Handles sparse data perfectly

When to use it:

Email spam

SMS spam

Any word-frequency–based text task

from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()

In [14]:
from sklearn.naive_bayes import MultinomialNB
model=MultinomialNB()
model.fit(X_train_res,y_train_res)

MultinomialNB()

In [15]:
y_predict=model.predict(X_test_vec)
acc=accuracy_score(y_test,y_predict)
acc

0.9895287958115183

In [16]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_predict))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       856
           1       0.99      0.97      0.98       290

    accuracy                           0.99      1146
   macro avg       0.99      0.98      0.99      1146
weighted avg       0.99      0.99      0.99      1146



In [18]:
#pkl
import pickle
with open("spam_classifier.pkl", "wb") as f:
    pickle.dump(model, f)   
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(text_vec, f)    
    